### Tools
Models can request to call tools that performs tasks such as fetching data from the database, searching the web, or running code. Tools are pairing of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function called coroutine to execute.

In [3]:
import os;
from langchain.chat_models import init_chat_model;
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY");
model = init_chat_model("gemini-3.5-flash", model_provider="google_genai");
response = model.invoke("What is the capital of France?");
print(response.content);

[{'type': 'text', 'text': 'The capital of France is **Paris**.', 'extras': {'signature': 'EssDCsgDAWkUfRPX2CQsrzPAdvkP6lYuZEkEUyK9uq/eVpXtaHK9lhJ2L7PCcRnakTs4Nd/BihS4izhs5lAJDldZwaExxDdc+79Da0OJYYymdSpNpSNPMiQw+BGPmY4dVyV58Nv4apyap2IQH1H5aTA9JQ9Jx91KhS1l5HTWDXRo3riHetP9rDoPtqRFngMZh6qdnj92G+bVflL/MRQnFZjvZzlWjIDTdnVi9MABkIWUDhgRk1IleokjjbhKufztBcPqbJtiCJJdAuTzS65Z+kNFXn+5DrXydgEESIcAS5dekZTnMx30jMGhum2C+1JK/idsO/DXsh4/tLscTOG6EBIBUYcRzloOX75217qlE4pAJZ3s2bJLQd81QAYKP4Ol8U1u15B3OVrIBg3Bf2EemkW8LQMuhM9E8EbcxWaEeJkhYKnjBymrS3Z/SGbIVvLqLqnk93nevxAZbz5SFDixMt9EkNfLWH+JyoZAhbBQefa5oIuDoNC/3PBhJrcym21XvTytFMb44YwhEO24qgkPLdXlNPhVRqFZxhBK7U/oZnCdyVlMX2H+fItn13kQnOvlHs7j+5ZIO8WiK5mIeea/BelH8Sis8YzuTzXRMAUJ'}}]


In [ ]:
from langchain.tools import tool;

@tool
def get_weather(city: str) -> str:
    """Get the weather of the city."""
    return f"The current weather in {city} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_weather]);

[]


In [7]:
response = model_with_tools.invoke("What is the weather in New York?");
print(response);

for tool_call in response.tool_calls:
    # View tool call made by model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_11702529385229455859': 'EvQCCvECAWkUfRMJSqixK5erMol8tXi91pQjqw3Jwnu/vEAj3+a59CAGSykEaUhzX8LfRrcHRkw2g+I1OFZXQukXDrYa4sPGmPgqg1ArrY2k7XFx+XNMyuvD/UaySYBxAtk2sQolZ98LMNZ0lN1R5FpVspsop1WGL58nmexRv9j74FSccu63gO6Im9unNTST/4I8WVG/HaWJT2KNhRNDODnQlam3GkocPXNhg2SrozxKS7yfEy8rVOFonHhQB49d+iPLrQJJxDF8UpL4o4i/o97NN9P7AaUQv5ilUGJP6SQNd2zP1A8PX/CSra89MLtuCjHWUeFXk7a5dyAoPr+u3mdqyB0vWHuwGW0al/gfQumYb/rQJxEUBSgPGCwO6BAHBy/RQVTab49o7TsTA57TLyi1uiOf+9slt/nFMYKw6oHHepD8m2nNAorTNjVHeRQ4o99qEotrvb3+bOodiTCs0tgUkj+0eHiXO4OZd6jda7cDoFC1NKtA'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'} id='lc_run--01a20711-b243-70be-86be-f5b2659dda32-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'call_1170

### Tool execution loops

In [8]:
# Step 1 - Model generates tool calls
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages);
messages.append(ai_msg);

# Step 2 - Execute tool calls and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call);
    messages.append(tool_result);

# Step 3 - Pass result back to model generates final response
final_response = model_with_tools.invoke(messages);
print(final_response.text);



The weather in New York is currently sunny with a temperature of 25°C.


In [9]:
messages

[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_15272682613538456821': 'EqwDCqkDAWkUfRMI95cQoU/iVfB9kOvcaqrT+I8IRcjOqJw1ptsAKvAiRLy6vR6MG1LdCr5MENQ5hDsEayyEVuhJRAs+0mJlElPZPOThffYIu28DfRzRErz25cQUZeATrIUTmpbU1ioa+kV8W9HltPShpTfjFka/ASFeS2DZHMQX68OZ8sNqG2YBGpIxvRq7GB/nw2dgba2vNHS3Bboyp1K/jAyvrjTOgdgo15e6ScOrqnp8Rh1OtUECFUfLNOqX1+Y5FYMvWDGhCnYhu6US6kUf34thts7toS2dviUdYRc11Td1+7SB/lC8D1iaB9v7dbbBGZ9aRm4P2VwBZuP4Lg2/yBw8XTrmR8upiIy4loPgbaVzPdQB9CpQ2qc3n277zK4Bf3x2hWiTJwgWamXp1K3sxAiMQ8hEqJH7KAaRAqBwOcomHBXQm2T4jSEgdjxiKpSmDujh4gHC53fi5JONOcCZRKWa+rw0jBLIvX3h5td+Wt11r4TAZM5PrEzm1waXAiEHl4DoeZSc3Uh4s08JFt6f5qqYtwH6f6cwCjQsvDE8g9h1g024QP26e7xb4vI='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provid